In [ ]:
import os
import pandas as pd
import mysql.connector
from dotenv import load_dotenv
from statsmodels.tsa.statespace.sarimax import SARIMAX
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Chargement des variables d'environnement depuis le fichier .env du dossier parent
load_dotenv(dotenv_path="../.env")

In [ ]:
# Connexion à la base de données Cloud Aiven MySQL
connection = mysql.connector.connect(
    host=os.getenv("AIVEN_HOST"),
    port=int(os.getenv("AIVEN_PORT", 14198)),
    user=os.getenv("AIVEN_USER"),
    password=os.getenv("AIVEN_PASSWORD"),
    database="defaultdb"
)

In [ ]:
# 1. Extraction d'un échantillon propre
# Note: On utilise la station 3012 (Saxe-Gambetta) ou 1034 qui contiennent des milliers d'enregistrements.
query = """
SELECT date_enregistrement AS date_enregistrement, velos 
FROM stations_historique 
WHERE id = 3012 
ORDER BY date_enregistrement ASC
"""

In [ ]:
# 2. Chargement dans un DataFrame
df = pd.read_sql(query, connection)
connection.close()

In [ ]:
# Vérification rapide du chargement
print("Colonnes chargées :", list(df.columns))
print("Nombre de lignes brutes :", len(df))
if len(df) > 0:
    print(df.head())

In [ ]:
# 3. Préparation des données (Index temporel + rééchantillonnage dynamique)
if 'date_enregistrement' in df.columns:
    df['date_enregistrement'] = pd.to_datetime(df['date_enregistrement'])
    df.set_index('date_enregistrement', inplace=True)

# Calcul de la durée totale des données chargées en heures
time_span_hours = (df.index.max() - df.index.min()).total_seconds() / 3600

# Choix dynamique de la fréquence de rééchantillonnage pour conserver assez de points (min 48)
if time_span_hours < 8:
    freq = '1T'  # Toutes les minutes
    print(f"Durée des données courte ({time_span_hours:.1f}h). Rééchantillonnage par minute ('1T')")
elif time_span_hours < 36:
    freq = '10T' # Toutes les 10 minutes
    print(f"Durée des données moyenne ({time_span_hours:.1f}h). Rééchantillonnage par 10 minutes ('10T')")
elif time_span_hours < 120:
    freq = '30T' # Toutes les 30 minutes
    print(f"Durée des données ({time_span_hours:.1f}h). Rééchantillonnage par 30 minutes ('30T')")
else:
    freq = 'h'   # Toutes les heures
    print(f"Durée des données longue ({time_span_hours:.1f}h). Rééchantillonnage par heure ('h')")

df = df.resample(freq).mean().ffill()
print(f"Nombre de points rééchantillonnés : {len(df)}")

# 4. Génération des graphiques de corrélation temporelle
# Ajustement dynamique du nombre de lags maximum par rapport aux données existantes
lags = min(48, len(df) // 2 - 1)

if lags >= 5:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))
    plot_acf(df['velos'], ax=ax1, lags=lags, title="ACF - Corrélation temporelle")
    plot_pacf(df['velos'], ax=ax2, lags=lags, title="PACF - Corrélation directe")
    plt.tight_layout()
    plt.show()
else:
    print(f"Pas assez de données pour générer l'ACF/PACF (lags calculés = {lags}, minimum requis = 5).")

In [ ]:
# 5. Configuration et ajustement du modèle SARIMAX ou ARIMA simple
if len(df) >= 10:
    # Si la durée totale est inférieure à 24 heures, on n'utilise pas de cycle saisonnier journalier
    if time_span_hours < 24:
        print("Entraînement d'un modèle ARIMA(1, 1, 1) simple (historique < 24h)...")
        model = SARIMAX(df['velos'], order=(1, 1, 1))
    else:
        # Calcul du cycle quotidien (S) en fonction de la fréquence choisie
        if freq == 'h': S = 24
        elif freq == '30T': S = 48
        elif freq == '10T': S = 144
        else: S = 24
        
        print(f"Entraînement du modèle SARIMAX(1, 1, 1)x(1, 1, 1, {S}) pour le cycle journalier...")
        model = SARIMAX(df['velos'], 
                        order=(1, 1, 1), 
                        seasonal_order=(1, 1, 1, S))
        
    results = model.fit()
    print(results.summary())
else:
    print("Pas assez de points d'observation pour ajuster le modèle (minimum 10 requis).")